In [601]:
# -- Downlaod locally
# https://huggingface.co/sentence-transformers/distiluse-base-multilingual-cased-v1/tree/main

# gif lfs install
# git config --global http.sslBackend schannel
# git clone https://huggingface.co/sentence-transformers/distiluse-base-multilingual-cased-v1

In [602]:
print("LangChain, Chroma, and Hugging Face form a powerful, entirely open-source tech stack commonly used to build Retrieval-Augmented Generation (RAG) systems")

LangChain, Chroma, and Hugging Face form a powerful, entirely open-source tech stack commonly used to build Retrieval-Augmented Generation (RAG) systems


In [603]:
# !pip install sentence-transformers
# !pip install langchain langchain-chroma langchain-huggingface langchain-community faiss-cpu sentence-transformers

In [604]:
import os
os.environ['CURL_CA_BUNDLE'] = ''

In [605]:
# # !pip install pysqlite3-binary 
# langchain-chroma error regarding pysqlite3
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [606]:
# SentenceTransformerEmbeddings: SBERT 모델을 활용한 문장 임베딩 생성
# from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
# FAISS: 효율적인 유사성 검색을 위한 벡터 데이터베이스 라이브러리
from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_openai import OpenAIEmbeddings
# from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import json

In [607]:
# 1. Load your local open-source data
# loader = TextLoader("your_data.txt")
# documents = loader.load()

In [608]:
def convert_documents_to_dataframe(docs):
    # Convert LangChain Documents to a list of dictionaries
    data = [
        {
            "page_content": doc.page_content,
            **doc.metadata  # Unpacks metadata keys directly into columns
        } 
        for doc in docs
    ]

    # print(data)

    # Create the Pandas DataFrame
    df = pd.DataFrame(data)
    display(df)

In [609]:
# 1. 문서 리스트 준비
sentences = [
    "우리나라는 2022년에 코로나가 유행했다.",
    "우리나라 2024년 GDP 전망은 3.0%이다.",
    "우리나라는 2022년 국내총생산 중 연구개발 예산은 약 5%이다.",
    "LangChain is a robust framework built to orchestrate LLM applications.",
    "Chroma is a highly efficient, open-source vector database tailored for AI applications.",
    "Hugging Face is a massive platform hosting thousands of open-source machine learning models."
]

In [610]:
# Wrap the strings into LangChain Document objects
documents = [Document(page_content=text, metadata=dict(page=i+1)) for i, text in enumerate(sentences)]

In [611]:
documents

[Document(metadata={'page': 1}, page_content='우리나라는 2022년에 코로나가 유행했다.'),
 Document(metadata={'page': 2}, page_content='우리나라 2024년 GDP 전망은 3.0%이다.'),
 Document(metadata={'page': 3}, page_content='우리나라는 2022년 국내총생산 중 연구개발 예산은 약 5%이다.'),
 Document(metadata={'page': 4}, page_content='LangChain is a robust framework built to orchestrate LLM applications.'),
 Document(metadata={'page': 5}, page_content='Chroma is a highly efficient, open-source vector database tailored for AI applications.'),
 Document(metadata={'page': 6}, page_content='Hugging Face is a massive platform hosting thousands of open-source machine learning models.')]

In [612]:
# Printout Dataframe format after traonsforming from Document
convert_documents_to_dataframe(documents)

[{'page_content': '우리나라는 2022년에 코로나가 유행했다.', 'page': 1}, {'page_content': '우리나라 2024년 GDP 전망은 3.0%이다.', 'page': 2}, {'page_content': '우리나라는 2022년 국내총생산 중 연구개발 예산은 약 5%이다.', 'page': 3}, {'page_content': 'LangChain is a robust framework built to orchestrate LLM applications.', 'page': 4}, {'page_content': 'Chroma is a highly efficient, open-source vector database tailored for AI applications.', 'page': 5}, {'page_content': 'Hugging Face is a massive platform hosting thousands of open-source machine learning models.', 'page': 6}]


,page_content,page
0,우리나라는 2022년에 코로나가 유행했다.,1
1,우리나라 2024년 GDP 전망은 3.0%이다.,2
2,우리나라는 2022년 국내총생산 중 연구개발 예산은 약 5%이다.,3
3,LangChain is a robust framework built to orche...,4
4,"Chroma is a highly efficient, open-source vect...",5
5,Hugging Face is a massive platform hosting tho...,6


In [613]:
# 2. Split documents into small, manageable chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

In [614]:
docs

[Document(metadata={'page': 1}, page_content='우리나라는 2022년에 코로나가 유행했다.'),
 Document(metadata={'page': 2}, page_content='우리나라 2024년 GDP 전망은 3.0%이다.'),
 Document(metadata={'page': 3}, page_content='우리나라는 2022년 국내총생산 중 연구개발 예산은 약 5%이다.'),
 Document(metadata={'page': 4}, page_content='LangChain is a robust framework built to orchestrate LLM applications.'),
 Document(metadata={'page': 5}, page_content='Chroma is a highly efficient, open-source vector database tailored for AI applications.'),
 Document(metadata={'page': 6}, page_content='Hugging Face is a massive platform hosting thousands of open-source machine learning models.')]

In [ ]:
# Printout Dataframe format after traonsforming from Document
convert_documents_to_dataframe(documents)

In [615]:
# 1. 임베딩 모델 준비
# embeddings = OpenAIEmbeddings()

# 2. SBERT 모델 초기화
# 사용 모델: distiluse-base-multilingual-cased-v1 (다국어 지원 모델)
# embedding = SentenceTransformerEmbeddings(model_name="distiluse-base-multilingual-cased-v1")

# 2. 임베딩 모델 로드 (한국어 지원 모델)
# 허깅페이스의 sbert-korean-base 모델을 사용합니다.
# embedding = SentenceTransformer('jhgan/ko-sroberta-multitask')

In [616]:

# 1. Download the model from Hugging Face
# model_name = "sentence-transformers/all-MiniLM-L6-v2"
# model = SentenceTransformer(model_name)

# 2. Save it to a local directory
# local_path = "./my_local_embedding_model"
# model.save(local_path)

# print(f"Model successfully saved offline to: {local_path}")

In [617]:
print("LangChain: The orchestrator framework. It coordinates document loading, splitting, vector database queries, and sending the final prompt to an LLM")
print("Hugging Face: The embedding and LLM provider. It supplies open-source text models")
print("Chroma (ChromaDB): The AI-native vector database. It stores those mathematical vectors locally or in the cloud and performs fast similarity searches")

LangChain: The orchestrator framework. It coordinates document loading, splitting, vector database queries, and sending the final prompt to an LLM
Hugging Face: The embedding and LLM provider. It supplies open-source text models
Chroma (ChromaDB): The AI-native vector database. It stores those mathematical vectors locally or in the cloud and performs fast similarity searches


In [618]:
print("Below is a complete, minimal script showcasing how to split text, embed it locally via Hugging Face, save it to a local Chroma database, and query it.")

Below is a complete, minimal script showcasing how to split text, embed it locally via Hugging Face, save it to a local Chroma database, and query it.


In [619]:
# --
# Hugging Face가 인터넷을 검색하지 않도록 오프라인 환경 변수 설정
# os.environ["HF_HUB_OFFLINE"] = "1"
# os.environ["TRANSFORMERS_OFFLINE"] = "1"

# 로컬에 저장된 모델 경로 불러오기
model_path = "../huggingfase_model/"

# model = SentenceTransformer(model_path)
# # 3. Calculate embeddings by calling model.encode()
# embeddings = model.encode(sentences, convert_to_numpy=True)
# print(f"Embeddings shape: {embeddings.shape}")

embedding_model = HuggingFaceEmbeddings(
    model_name=model_path
)
print(f"Embeddings: {embedding_model}")
# --

Embeddings: model_name='../huggingfase_model/' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [620]:

# - doc_list의 각 문장을 벡터화하여 FAISS 데이터베이스에 저장
# - 각 문서에 메타데이터를 부여하여 추적 가능하도록 설정
# vector_store = FAISS.from_texts(
#     sentences, embeddings, metadatas=[{"source": 1}] * len(doc_list)
# )

# Step 3: Create the FAISS index and add the documents
print("Generating embeddings and building FAISS index...")
# Step 3: Create and populate the Chroma Vector Database
# vector_store = FAISS.from_documents(docs, embedding_model)
vector_store = Chroma.from_documents(documents=docs, embedding=embedding_model, persist_directory="./chroma_db")
print("Index successfully created!")

Generating embeddings and building FAISS index...
Index successfully created!


In [621]:
# 5. 검색 쿼리 입력
# query = "2022년 우리나라 GDP대비 R&D 규모는?"
query = "What is the tool used for storing vectors?"

In [622]:
docs = vector_store.similarity_search(query, k=5)
print(docs[0].page_content)

Chroma is a highly efficient, open-source vector database tailored for AI applications.


In [623]:
# 4. FAISS 검색기(FaissRetriever) 설정
# - k=1: 가장 유사한 문장 1개만 검색
vector_store = vector_store.as_retriever(search_kwargs={"k": 5})

In [624]:
# 6. 유사 문서 검색 수행
docs = vector_store.invoke(query)

In [625]:
# 7. 검색 결과 출력
print(docs)
# 결과: "우리나라는 2022년 국내총생산 중 연구개발 예산은 약 5%이다." 문장이 가장 유사한 결과로 반환

[Document(id='b75dcf22-7dd3-415b-8ca4-b48a7dbe5008', metadata={'page': 5}, page_content='Chroma is a highly efficient, open-source vector database tailored for AI applications.'), Document(id='ecad1a54-88b3-49df-b264-8445e3690faa', metadata={'page': 5}, page_content='Chroma is a highly efficient, open-source vector database tailored for AI applications.'), Document(id='6de88b0b-79ef-4db7-a077-936540db1d29', metadata={'page': 5}, page_content='Chroma is a highly efficient, open-source vector database tailored for AI applications.'), Document(id='3da3f947-34bf-4cdc-a1be-808948dd2287', metadata={'page': 5}, page_content='Chroma is a highly efficient, open-source vector database tailored for AI applications.'), Document(id='8232e8c0-a9f3-461b-b6af-ee63e2a778ef', metadata={'page': 5}, page_content='Chroma is a highly efficient, open-source vector database tailored for AI applications.')]


In [ ]:
# Printout Dataframe format after traonsforming from Document
convert_documents_to_dataframe(docs)

In [627]:
# Step 6: Print out the best match
print("Top Match Content:")
print(docs[0].page_content)

Top Match Content:
Chroma is a highly efficient, open-source vector database tailored for AI applications.
